<h2 align='center'>Codebasics ML Course: ML Flow Tutorial</h2>

In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Step 1: Create an imbalanced binary classification dataset
X, y = make_classification(n_samples=1000, n_features=10, n_informative=2, n_redundant=8, 
                           weights=[0.9, 0.1], flip_y=0, random_state=42)

np.unique(y, return_counts=True)

(array([0, 1]), array([900, 100]))

In [3]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

#### Handle class imbalance

In [4]:
from imblearn.combine import SMOTETomek

smt = SMOTETomek(random_state=42)
X_train_res, y_train_res = smt.fit_resample(X_train, y_train)
np.unique(y_train_res, return_counts=True)

(array([0, 1]), array([619, 619]))

### Track Experiments

In [5]:
models = [
    (
        "Logistic Regression", 
        {"C": 1, "solver": 'liblinear'},
        LogisticRegression(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "Random Forest", 
        {"n_estimators": 30, "max_depth": 3},
        RandomForestClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier With SMOTE",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(), 
        (X_train_res, y_train_res),
        (X_test, y_test)
    )
]

In [6]:
reports = []

for model_name, params, model, train_set, test_set in models:
    X_train = train_set[0]
    y_train = train_set[1]
    X_test = test_set[0]
    y_test = test_set[1]
    
    model.set_params(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)
    reports.append(report)

In [7]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost

In [8]:
import dagshub
dagshub.init(repo_owner='sawankr1987', repo_name='ml-flow-dagshub', mlflow=True)

Accessing as sawankr1987

Initialized MLflow to track repo "sawankr1987/ml-flow-dagshub"

Repository sawankr1987/ml-flow-dagshub initialized!

In [14]:
import mlflow

# Initialize MLflow
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Anomaly Detection")

for i, element in enumerate(models):
    model_name = element[0]
    params = element[1]
    model = element[2]
    report = reports[i]  # expected: classification_report(..., output_dict=True)

    with mlflow.start_run(run_name=model_name):
        # --- Params ---
        mlflow.log_params(params)

        # --- Metrics (safe extraction + casting) ---
        metrics = {}

        # accuracy (top-level)
        metrics["accuracy"] = float(report.get("accuracy", 0.0))

        # class-wise recall (guard for missing classes)
        if "1" in report:
            metrics["recall_class_1"] = float(report["1"].get("recall", 0.0))
        if "0" in report:
            metrics["recall_class_0"] = float(report["0"].get("recall", 0.0))

        # macro F1
        if "macro avg" in report:
            metrics["f1_score_macro"] = float(report["macro avg"].get("f1-score", 0.0))

        # (optional) weighted F1
        if "weighted avg" in report:
            metrics["f1_score_weighted"] = float(report["weighted avg"].get("f1-score", 0.0))

        mlflow.log_metrics(metrics)

        # --- Model logging ---
        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(model, "model")

2026/04/18 18:19:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/18 18:19:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Logistic Regression at: http://localhost:5000/#/experiments/2/runs/f6a20292aa2f4102a33d0d12b4f33bc0
🧪 View experiment at: http://localhost:5000/#/experiments/2


2026/04/18 18:19:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/18 18:19:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Random Forest at: http://localhost:5000/#/experiments/2/runs/16ba57f0f14d40c1b55a0cbe8924aff0
🧪 View experiment at: http://localhost:5000/#/experiments/2


2026/04/18 18:20:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: http://localhost:5000/#/experiments/2/runs/176429d5e6b24cb88fe10acd6e818b31
🧪 View experiment at: http://localhost:5000/#/experiments/2


2026/04/18 18:20:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier With SMOTE at: http://localhost:5000/#/experiments/2/runs/414a8e5e019142858ad9ae29df0d1882
🧪 View experiment at: http://localhost:5000/#/experiments/2


### Register the Model

In [15]:
model_name = "XGB-Smote"
run_id = input("Enter Run ID: ")

model_uri = f"runs:/{run_id}/model"

mlflow.register_model(model_uri=model_uri, name=model_name)

Registered model 'XGB-Smote' already exists. Creating a new version of this model...
2026/04/18 18:26:39 WARNING mlflow.tracking._model_registry.fluent: Run with id 414a8e5e019142858ad9ae29df0d1882 has no artifacts at artifact path 'model', registering model based on models:/m-ece82ca6ebcd42cf9feefd120fd86d86 instead
2026/04/18 18:26:40 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGB-Smote, version 1
Created version '1' of model 'XGB-Smote'.


<ModelVersion: aliases=[], creation_timestamp=1776517000150, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1776517000150, metrics=None, model_id=None, name='XGB-Smote', params=None, run_id='414a8e5e019142858ad9ae29df0d1882', run_link='', source='models:/m-ece82ca6ebcd42cf9feefd120fd86d86', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>

### Load the Model

In [16]:
model_version = 1
model_uri = f"models:/{model_name}/{model_version}"

loaded_model = mlflow.xgboost.load_model(model_uri)
y_pred = loaded_model.predict(X_test)
y_pred[:4]

array([0, 0, 0, 0])

In [19]:
client = mlflow.MlflowClient()

client.set_registered_model_alias(
    name=model_name,
    alias="challenger",
    version=1
)

### Transition the Model to Production

In [20]:
current_model_uri = f"models:/{model_name}@challenger"
production_model_name = "anomaly-detection-prod"

client = mlflow.MlflowClient()
client.copy_model_version(src_model_uri=current_model_uri, dst_name=production_model_name)

Successfully registered model 'anomaly-detection-prod'.
Copied version '1' of model 'XGB-Smote' to version '1' of model 'anomaly-detection-prod'.


<ModelVersion: aliases=[], creation_timestamp=1776517394864, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1776517394864, metrics=None, model_id=None, name='anomaly-detection-prod', params=None, run_id='414a8e5e019142858ad9ae29df0d1882', run_link='', source='models:/XGB-Smote/1', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>

In [22]:
client = mlflow.MlflowClient()

client.set_registered_model_alias(
    name=production_model_name,
    alias="champion",
    version=1
)

In [23]:
model_version = 1
prod_model_uri = f"models:/{production_model_name}@champion"

loaded_model = mlflow.xgboost.load_model(prod_model_uri)
y_pred = loaded_model.predict(X_test)
y_pred[:4]

array([0, 0, 0, 0])

In [ ]:
import os
import mlflow

# Set tracking FIRST
os.environ['MLFLOW_TRACKING_USERNAME'] = 'your_username'
os.environ['MLFLOW_TRACKING_PASSWORD'] = 'your_token'
os.environ['MLFLOW_TRACKING_URI'] = "https://dagshub.com/sawankr1987/ml-flow-dagshub.mlflow"

# OR explicitly (recommended)
mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])

# THEN set experiment
mlflow.set_experiment("Anomaly Detection")

<Experiment: artifact_location='mlflow-artifacts:/37af96c8e5de49d3967da762f08ae4d3', creation_time=1776513632731, experiment_id='0', last_update_time=1776513632731, lifecycle_stage='active', name='Anomaly Detection', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

Please refer to following to learn more about model registry

https://mlflow.org/docs/latest/model-registry.html#model-registry-workflows to learn 